## Prediction Workflow
This notebook demonstrates how the results from the previous notebooks can be used in the prediction workflow.
The prediction workflow is shown using the instance segmentation model for hyperbolas, which was the approach that allowed successful hyperbola fitting.
This notebook also serves as a user guide on how to apply the prediction workflow.
It follows these steps:

1. **Load the SGY File with the DataToolKit class**
    - Create the Dataframe (for creating correct images and also physical calculations at the end)
    - Creating Images for prediction
2. **Prediction with Predictor Class**
    - using the YOLO Model for Prediction
    - Matching Detections (bringing  them into 3D Context)
3. **Fitting the Hyperbolas with Predictor Class**
4. **Plotting the Results with Predictor Class**

First Setting Working directory and load config file.

In [13]:
#Importing used Packages to load the File
import sys
from pathlib import Path

#Setting Working directory
sys.path.append(str(Path.cwd().parent))

#Import the config file so that only the Filename needs to be changed in the _read_segy function
from config import *

### Step1: Loading SGY File
For this step, the DataToolKit class is used.
Note: SGY files are too large for GitHub. Please copy them into /Data/Testdata/Files.

In [14]:
from Pipeline.Datatoolkit import DatatoolKit
dk = DatatoolKit(TEST_FILE_DIR, "UG3DQUERUNTERZUG.SGY")

In [15]:
sgy_file = dk.LoadSGY()

Next step is to create a Dataframe so that Images can be created for Inference.

In [16]:
df_sgy = dk.create_df(file=sgy_file)

next is to create images for prediction her the inline cuts are created there is also a function implemented which creates randomly a number of images. The random function could be useful to make some quick tests.

In [17]:
dk.create_images(file= sgy_file, df = df_sgy, outdir = TEST_PIC_SEG_H_DIR, inline= True)

saved 60 images to c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_60.png


### Step 2: Prediction with Predictor Class

First the import of the Predictor class is made. and the ultralytics library is importet for Prediction

In [18]:
from ultralytics import YOLO
from Pipeline.Predictor import Predictor
p = Predictor()

Next step is to load the trained Model and make Predictions on the created images

In [19]:
model = YOLO(YOLO_MODEL_SEG_H_DIR)
results = model.predict(TEST_PIC_SEG_H_DIR, save = True, conf= 0.2, show_boxes = True)


image 1/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_1.png: 384x640 16 hyperbolas, 65.0ms
image 2/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_10.png: 384x640 18 hyperbolas, 59.7ms
image 3/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_11.png: 384x640 18 hyperbolas, 57.5ms
image 4/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_12.png: 384x640 20 hyperbolas, 53.9ms
image 5/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_13.png: 384x640 20 hyperbolas, 59.6ms
image 6/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbola\pictures\UG3DQUERUNTERZUG.SGY_inline_14.png: 384x640 19 hyperbolas, 67.9ms
image 7/60 c:\pythonad\PAINDHS25\PAINDGPR\Data\Testdata\Segmentation_Hyperbo

The Next step is to match the detections which means get them into the 3D Context and additonally saving a csv.

In [20]:
matched_detections = p.match_detections(results=results, dist_trheshhold=20, save_path=OUT_PATH_Detections, export = True)

lets see if all detections were captured in the matched_detections

In [21]:
p.validate_detections(matched_detections=matched_detections, results=results)

Detections YOLO matching with Matched Detections DF


### Step 3: Fitting Hyperbolas

fitting the Hyperbolas based on the matched detections and saving a csv. there are 2 Fitting procedures
- Fitting per  cut
- idealized
Here the fitting is done for both

In [22]:
idealized_fit, fits_per_cut = p.fit_hyperbolas(matched_detections=matched_detections, export_csv= True, save_path=OUT_PATH_Detections)

idealized fit (fit for bbox: 0, coefficients[ 6.2691e+05     -2143.7      1.8339])
idealized fit (fit for bbox: 1, coefficients[      39596     -539.76      1.8643])
idealized fit (fit for bbox: 2, coefficients[ 4.8715e+05     -1836.9       1.733])
idealized fit (fit for bbox: 3, coefficients[ 3.2576e+05     -1348.3      1.3971])
idealized fit (fit for bbox: 4, coefficients[ 6.8414e+05     -2424.8      2.1645])
idealized fit (fit for bbox: 5, coefficients[ 2.5041e+05     -1180.6      1.3942])
idealized fit (fit for bbox: 6, coefficients[ 9.9221e+05     -3270.4      2.7067])
idealized fit (fit for bbox: 7, coefficients[      14840     -326.47      1.8649])
idealized fit (fit for bbox: 8, coefficients[   8.59e+05     -2446.8      1.7433])
idealized fit (fit for bbox: 9, coefficients[ 1.1035e+06     -3291.6       2.464])
idealized fit (fit for bbox: 10, coefficients[      49922      -769.1      3.4896])
idealized fit (fit for bbox: 11, coefficients[     473.16      6.4169      1.4186])
id

### Step 4:Plotting the results and convert imprtant information to Physical 

first convert important information into physical dimension because fitting is done on Pixel Values

In [23]:
(number_of_crosslines, 
 number_of_inlines, 
 sampels_per_trace, 
 sample_rate) = p.get_axis_and_sample_rate(sgy_file=sgy_file , df=df_sgy)

#factor 100 because sample_rate is in picoseconds widt_inlines and crosslines are taken from videos from industrypartner but can set individually
(time_per_pixel, 
 distance_per_pixel_crosslines, 
 distance_per_pixel_inlines) = p.help_function_physical_units(sample_rate= sample_rate, 
                                                              sample_rate_factor=100, 
                                                              number_of_crosslines=number_of_crosslines, 
                                                              number_of_inlines= number_of_inlines, 
                                                              width_crosslines=4,
                                                              width_inlines=3)

#calculate the parameters t0,v,x0 in Physical units and add them to the df and save them as csv
idealized_fit_phys = p.convert_phys_params(df=idealized_fit, 
                                           dcrosslines=distance_per_pixel_crosslines, 
                                           dtime= time_per_pixel, 
                                           export_csv=True,
                                           name_of_df="idealized_fit", 
                                           save_path=OUT_PATH_Detections)

fits_per_cut_phys = p.convert_phys_params(df= fits_per_cut, 
                                          dcrosslines= distance_per_pixel_crosslines, 
                                          dtime=time_per_pixel, export_csv=True,
                                          name_of_df="fits_per_cut", 
                                          save_path=OUT_PATH_Detections)



The Plot uses the functions from above when physics_enabled and the parameter width_crosslines and width_inlines are given. The function also saves an html File and plots the interactive plot immediatly in a webbrowser.

In [24]:

p.plot_hyperbolas_3d_interactive(fits_per_cut_df=fits_per_cut, 
                                 fit_idealized_df=idealized_fit,
                                 matched_detections=matched_detections,
                                 physics_enabled=True, 
                                 sgy_file=sgy_file, 
                                 df_from_DatatoolKit=df_sgy, 
                                 width_crosslines=4, 
                                 width_inlines=3, 
                                 sample_rate_factor=1000)